# Final Project, Part 4 Resubmission

Group Members:

Mianchen Zhang, Miantong Zhang

In [1]:
import pandas as pd
import altair as alt

In [2]:
expenditures_df = pd.read_csv("City_Of_Urbana__Open_Expenditures_-_Payment_Details_20260413.csv", low_memory=False)
permits_df = pd.read_csv("City_of_Urbana__Community_Development_Permits_Issued_20260510.csv", low_memory=False)

main_df = expenditures_df.copy()
context_df = permits_df.copy()

In [3]:
# clean datasets
main_df["fiscal_year"] = (
    main_df["fiscal_year"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

main_df["fiscal_year"] = pd.to_numeric(main_df["fiscal_year"], errors="coerce")

main_df["amount"] = (
    main_df["amount"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

main_df["amount"] = pd.to_numeric(main_df["amount"], errors="coerce")
main_df["payment_date"] = pd.to_datetime(
    main_df["payment_date"],
    format="%Y %b %d %I:%M:%S %p",
    errors="coerce"
)
main_df["invoice_date"] = pd.to_datetime(
    main_df["invoice_date"],
    format="%Y %b %d %I:%M:%S %p",
    errors="coerce"
)

main_df["department"] = main_df["department"].fillna("Unknown").str.strip()
main_df["division"] = main_df["division"].fillna("Unknown").str.strip()
main_df["vendor"] = main_df["vendor"].fillna("Unknown").str.strip()

main_df["department"] = main_df["department"].replace({
    "NOT APPLICABLE": "Unknown"
})

main_df["vendor"] = main_df["vendor"].replace({
    "NOT AVAILABLE": "Unknown",
    "Not Available": "Unknown",
    "VENDOR NAME NOT AVAILABLE": "Unknown"
})

main_df = main_df.dropna(subset=["fiscal_year", "amount"])

context_df["Issue Year"] = (
    context_df["Issue Year"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

context_df["Issue Year"] = pd.to_numeric(context_df["Issue Year"], errors="coerce")
context_df["Issue Date"] = pd.to_datetime(
    context_df["Issue Date"],
    format="%Y %b %d %I:%M:%S %p",
    errors="coerce"
)

for col in ["Estimated Cost", "Fee Amount", "Square Feet"]:
    context_df[col] = (
        context_df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    context_df[col] = pd.to_numeric(context_df[col], errors="coerce")

for col in ["Permit Type", "Improvement Type", "Building Type", "Applicant", "Permit Status"]:
    context_df[col] = context_df[col].fillna("Unknown").str.strip()

context_df["Improvement Type"] = context_df["Improvement Type"].replace({
    "Not Available on this Old Permit": "Unknown"
})

context_df = context_df.dropna(subset=["Issue Year", "Issue Date"])

dept_clean = main_df[main_df["department"] != "Unknown"]

vendor_clean = main_df[main_df["vendor"] != "Unknown"]

### Overall Spending by Department (Visualization 1)

In [4]:
dept = dept_clean.groupby("department", as_index=False)["amount"].sum()

dept = dept.sort_values("amount", ascending=False).head(15)
dept

,department,amount
13,PUBLIC WORKS,1.892915e+08
3,COMMUNITY DEVELOPMENT,1.126066e+08
11,POLICE,7.655707e+07
5,FINANCE,4.238596e+07
7,FIRE AND RESCUE,4.186000e+07
9,GENERAL SERVICES,3.283758e+07
4,EXECUTIVE,1.894025e+07
6,FIRE,1.458580e+07
2,CITY-WIDE ACTIVITIES,4.549546e+06
10,HUMAN RESOURCES & FINANCE DEPT,3.159813e+06


In [6]:
bars = alt.Chart(dept).mark_bar().encode(
    x=alt.X(
        "amount:Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    y=alt.Y(
        "department:N",
        sort="-x",
        title="Department"
    ),
    color=alt.condition(
        "datum.department === 'PUBLIC WORKS' || datum.department === 'COMMUNITY DEVELOPMENT'",
        alt.value("#D95F02"),
        alt.value("#C9C9C9")
    ),
    tooltip=[
        alt.Tooltip("department:N", title="Department"),
        alt.Tooltip("amount:Q", title="Total Spending", format="$,.2f")
    ]
).properties(
    width=700,
    height=400,
    title="Top Departments by Total Spending"
)

bars

alt.Chart(...)

Public Works and Community Development deserve special attention because they account for the largest spending totals while representing different public priorities. Public Works reflects the cost of maintaining streets, facilities, utilities, and other basic infrastructure that residents depend on every day, so changes in this department can affect the city broadly and immediately. Community Development is important for a different reason: it captures spending connected to housing, redevelopment, and neighborhood improvement, which shapes longer-term growth and quality of life. Together, these two departments highlight both the city’s day-to-day operating responsibilities and its longer-term investment strategy.

### Development-Related Spending Over Time (Visualization 2)

In [7]:
yearly = main_df.groupby("fiscal_year", as_index=False)["amount"].sum()

development_departments = ["COMMUNITY DEVELOPMENT", "PUBLIC WORKS"]

development_yearly = (
    main_df[main_df["department"].isin(development_departments)]
    .groupby(["fiscal_year", "department"], as_index=False)["amount"]
    .sum()
    .sort_values(["fiscal_year", "department"])
)

development_yearly

,fiscal_year,department,amount
0,2013,COMMUNITY DEVELOPMENT,10300661.61
1,2013,PUBLIC WORKS,16616273.56
2,2014,COMMUNITY DEVELOPMENT,24735855.04
3,2014,PUBLIC WORKS,18955191.31
4,2015,COMMUNITY DEVELOPMENT,20304467.70
5,2015,PUBLIC WORKS,19874291.12
6,2016,COMMUNITY DEVELOPMENT,7342489.90
7,2016,PUBLIC WORKS,18822291.66
8,2017,COMMUNITY DEVELOPMENT,8224274.82
9,2017,PUBLIC WORKS,16443222.58


In [8]:
development_line = alt.Chart(development_yearly).mark_line(point=True, strokeWidth=3).encode(
    x=alt.X(
        "fiscal_year:O",
        title="Fiscal Year",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "amount:Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    color=alt.Color(
        "department:N",
        title="Department",
        scale=alt.Scale(
            domain=["PUBLIC WORKS", "COMMUNITY DEVELOPMENT"],
            range=["#D95F02", "#1B9E77"]
        )
    ),
    tooltip=[
        alt.Tooltip("fiscal_year:O", title="Year"),
        alt.Tooltip("department:N", title="Department"),
        alt.Tooltip("amount:Q", title="Total Spending", format="$,.2f")
    ]
).properties(
    width=720,
    height=400,
    title="Yearly Spending Trends for Community Development and Public Works"
).configure_title(
    fontSize=18,
    anchor="middle"
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
).interactive()

line = development_line
line

alt.Chart(...)

This chart focuses on Community Development and Public Works because they represent two important but different kinds of development-related spending. Public Works stays above Community Development in most years, suggesting that infrastructure and maintenance create a larger and more consistent budget commitment. Community Development is more volatile, with especially large spikes in 2014, 2015, and 2025, which may reflect periods of concentrated redevelopment, housing, or neighborhood investment. Looking at these two departments together helps separate steady infrastructure spending from more project-driven development activity.

### Transition to Context Dataset

Spending trends alone cannot explain actual development activity. A department may spend more money in a given year, but that does not automatically show how many projects were started, what kinds of construction were taking place, or whether development activity increased across the city. To add that missing context, I use the Urbana permits dataset, which records permit issuance related to building and other development work. This context dataset makes it possible to connect financial patterns in the expenditures data to evidence of real development activity on the ground.

### Community Development Permits Over Time (Visualization 3)

In [9]:
permits_yearly = (
    context_df.groupby("Issue Year")["Permit Number"]
    .nunique()
    .reset_index(name="permits_issued")
    .rename(columns={"Issue Year": "issue_year"})
    .sort_values("issue_year")
)

permits_yearly["issue_year"] = permits_yearly["issue_year"].astype(int)

permits_line = alt.Chart(permits_yearly).mark_line(
    point=True,
    strokeWidth=3,
    color="#2A9D8F"
).encode(
    x=alt.X(
        "issue_year:O",
        title="Issue Year",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "permits_issued:Q",
        title="Permits Issued",
        axis=alt.Axis(format=",.0f")
    ),
    tooltip=[
        alt.Tooltip("issue_year:O", title="Year"),
        alt.Tooltip("permits_issued:Q", title="Permits Issued", format=",.0f")
    ]
).properties(
    width=720,
    height=400,
    title="Community Development Permits Issued per Year"
).configure_title(
    fontSize=18,
    anchor="middle"
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
).interactive()

permits_line

alt.Chart(...)

This chart uses the permits dataset to track development activity more directly than spending alone. Permit issuance was higher in the early 2000s, declined through the late 2000s and early 2010s, and then strengthened again after 2021, exceeding 2,000 permits in both 2023 and 2024. That rebound suggests recent development activity has picked up even when expenditure totals are harder to interpret by themselves. The drop in 2026 should be read cautiously because the file appears to have been downloaded on May 10, 2026, so the 2026 count likely reflects only part of the year.

### Spending vs Development Activity Comparison (Visualization 4)

In [10]:
community_spending_yearly = (
    main_df[main_df["department"] == "COMMUNITY DEVELOPMENT"]
    .groupby("fiscal_year", as_index=False)["amount"]
    .sum()
    .rename(columns={"fiscal_year": "Year", "amount": "RawValue"})
)

permits_activity_yearly = (
    context_df.groupby("Issue Year")["Permit Number"]
    .nunique()
    .reset_index(name="RawValue")
    .rename(columns={"Issue Year": "Year"})
)

comparison_wide = (
    community_spending_yearly.rename(columns={"RawValue": "SpendingRawValue"})
    .merge(
        permits_activity_yearly.rename(columns={"RawValue": "PermitsRawValue"}),
        on="Year",
        how="inner"
    )
    .sort_values("Year")
)

common_years = comparison_wide["Year"].astype(int).tolist()
baseline_year = 2013 if 2013 in common_years else min(common_years)

comparison_wide = comparison_wide[
    (comparison_wide["Year"] >= baseline_year) &
    (comparison_wide["Year"] <= 2025)
].copy()
comparison_wide["Year"] = comparison_wide["Year"].astype(int)

comparison_frames = []

for metric, raw_column, formatter in [
    ("Community Development Spending", "SpendingRawValue", lambda x: f"${x:,.0f}"),
    ("Permits Issued", "PermitsRawValue", lambda x: f"{x:,.0f}")
]:
    temp = comparison_wide[["Year", raw_column]].rename(columns={raw_column: "RawValue"}).copy()
    temp["Metric"] = metric
    baseline_value = temp.loc[temp["Year"] == baseline_year, "RawValue"].iloc[0]
    temp["IndexValue"] = (temp["RawValue"] / baseline_value) * 100
    temp["RawValueLabel"] = temp["RawValue"].map(formatter)
    comparison_frames.append(temp)

comparison_long = pd.concat(comparison_frames, ignore_index=True)

hover = alt.selection_point(fields=["Year"], nearest=True, on="mouseover", empty=False)

baseline_rule = alt.Chart(pd.DataFrame({"Baseline": [100]})).mark_rule(
    color="#8A8A8A",
    strokeDash=[5, 4]
).encode(
    y="Baseline:Q"
)

comparison_base = alt.Chart(comparison_long).encode(
    x=alt.X(
        "Year:O",
        sort="ascending",
        title="Year",
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "IndexValue:Q",
        title=f"Indexed Relative Change ({baseline_year} = 100)",
        axis=alt.Axis(format=".0f", gridColor="#E6E6E6", gridOpacity=0.8)
    ),
    color=alt.Color(
        "Metric:N",
        title=None,
        legend=alt.Legend(orient="top-right", title=None, labelFontSize=12, symbolSize=160, labelLimit=320),
        scale=alt.Scale(
            domain=["Community Development Spending", "Permits Issued"],
            range=["#1F77B4", "#D55E00"]
        )
    ),
    tooltip=[
        alt.Tooltip("Year:O", title="Year"),
        alt.Tooltip("Metric:N", title="Metric"),
        alt.Tooltip("RawValueLabel:N", title="Raw Value"),
        alt.Tooltip("IndexValue:Q", title="Indexed Value", format=".1f")
    ]
)

comparison_lines = comparison_base.mark_line(point=True, strokeWidth=3)
comparison_hover_points = comparison_base.mark_circle(size=90).transform_filter(hover)
comparison_hover_rule = alt.Chart(comparison_long).mark_rule(color="#BDBDBD").encode(
    x="Year:O"
).transform_filter(hover)

comparison_chart = (
    baseline_rule + comparison_lines + comparison_hover_points + comparison_hover_rule
).add_params(
    hover
).properties(
    width=980,
    height=450,
    title="Development Activity and Public Spending Trends in Urbana"
).configure_title(
    fontSize=18,
    anchor="middle"
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
).configure_legend(
    labelFontSize=12,
    titleFontSize=12
).interactive()

comparison_chart

alt.LayerChart(...)

To better compare the two datasets despite their different units, both Community Development spending and permit activity were converted into indexed relative values using 2013 as the baseline year (2013 = 100). This allows the visualization to focus on relative change over time rather than raw dollar amounts or permit counts.

The indexed chart shows that Community Development spending experienced a sharp early spike, especially around 2014, but later remained below the 2013 baseline for much of the period. In contrast, permit activity gradually increased after the baseline year and rose especially strongly after 2021. This divergence suggests that Community Development spending does not directly track the number of permits issued each year. Instead, spending may reflect longer-term planning cycles, staffing or administrative costs, infrastructure preparation, maintenance needs, inflation, or delayed responses to development activity. Overall, the comparison suggests that Urbana’s spending relates to community development activity, but not in a simple one-to-one way.

### What Drives Development-Related Spending in Urbana?

In [11]:
development_departments = ["COMMUNITY DEVELOPMENT", "PUBLIC WORKS"]

dashboard_source = main_df[main_df["department"].isin(development_departments)].copy()
dashboard_source["Year"] = dashboard_source["fiscal_year"].astype(int)
dashboard_source["expense_category"] = dashboard_source["expense_category"].fillna("Unknown").str.strip()
dashboard_source["vendor"] = dashboard_source["vendor"].fillna("Unknown").str.strip()
dashboard_source["vendor"] = dashboard_source["vendor"].replace({
    "NOT AVAILABLE": "Unknown",
    "Not Available": "Unknown",
    "VENDOR NAME NOT AVAILABLE": "Unknown"
})

dashboard_source["payment_id_str"] = dashboard_source["payment_id"].astype(str)
dashboard_source.loc[dashboard_source["payment_id"].isna(), "payment_id_str"] = (
    "missing_" + dashboard_source.loc[dashboard_source["payment_id"].isna()].index.astype(str)
)

dashboard_detail = dashboard_source.copy()
dashboard_detail["DepartmentOption"] = dashboard_detail["department"]

dashboard_all = dashboard_source.copy()
dashboard_all["DepartmentOption"] = "All Development-Related Departments"

dashboard_long = pd.concat([dashboard_detail, dashboard_all], ignore_index=True)

def shorten_text(text, limit=34):
    text = str(text)
    return text if len(text) <= limit else text[: limit - 1] + "…"

category_breakdown = (
    dashboard_long.groupby(["Year", "DepartmentOption", "expense_category"], as_index=False)
    .agg(
        TotalSpending=("amount", "sum"),
        PaymentCount=("payment_id_str", "nunique")
    )
    .rename(columns={"expense_category": "Category"})
    .sort_values(["Year", "DepartmentOption", "TotalSpending"], ascending=[True, True, False])
)
category_breakdown["Rank"] = (
    category_breakdown.groupby(["Year", "DepartmentOption"])["TotalSpending"]
    .rank(method="first", ascending=False)
)
category_top10 = category_breakdown[category_breakdown["Rank"] <= 10].copy()

top_category_summary = (
    category_breakdown.groupby(["Year", "DepartmentOption"], as_index=False)
    .first()[["Year", "DepartmentOption", "Category"]]
    .rename(columns={"Category": "TopCategoryFull"})
)

vendor_breakdown = (
    dashboard_long[dashboard_long["vendor"] != "Unknown"]
    .groupby(["Year", "DepartmentOption", "vendor"], as_index=False)
    .agg(
        TotalSpending=("amount", "sum"),
        PaymentCount=("payment_id_str", "nunique")
    )
    .rename(columns={"vendor": "Vendor"})
    .sort_values(["Year", "DepartmentOption", "TotalSpending"], ascending=[True, True, False])
)
vendor_breakdown["Rank"] = (
    vendor_breakdown.groupby(["Year", "DepartmentOption"])["TotalSpending"]
    .rank(method="first", ascending=False)
)
vendor_top10 = vendor_breakdown[vendor_breakdown["Rank"] <= 10].copy()

top_vendor_summary = (
    vendor_breakdown.groupby(["Year", "DepartmentOption"], as_index=False)
    .first()[["Year", "DepartmentOption", "Vendor"]]
    .rename(columns={"Vendor": "TopVendorFull"})
)

summary_df = (
    dashboard_long.groupby(["Year", "DepartmentOption"], as_index=False)
    .agg(
        TotalSpending=("amount", "sum"),
        PaymentCount=("payment_id_str", "nunique")
    )
    .merge(top_category_summary, on=["Year", "DepartmentOption"], how="left")
    .merge(top_vendor_summary, on=["Year", "DepartmentOption"], how="left")
)

summary_df["TopVendorFull"] = summary_df["TopVendorFull"].fillna("Unavailable")
summary_df["TotalSpendingDisplay"] = summary_df["TotalSpending"].map(lambda x: f"${x:,.0f}")
summary_df["PaymentCountDisplay"] = summary_df["PaymentCount"].map(lambda x: f"{x:,}")
summary_df["TopCategoryDisplay"] = summary_df["TopCategoryFull"].map(lambda x: shorten_text(x, 32))
summary_df["TopVendorDisplay"] = summary_df["TopVendorFull"].map(lambda x: shorten_text(x, 32))

dashboard_year_options = sorted(summary_df["Year"].unique().tolist())
default_dashboard_year = 2025 if 2025 in dashboard_year_options else max(dashboard_year_options)
dashboard_department_options = [
    "All Development-Related Departments",
    "COMMUNITY DEVELOPMENT",
    "PUBLIC WORKS"
]


In [12]:
year_filter = alt.selection_point(
    name="year_filter",
    fields=["Year"],
    bind=alt.binding_select(options=dashboard_year_options, name="Year: "),
    value=[{"Year": default_dashboard_year}]
)

department_filter = alt.selection_point(
    name="department_filter",
    fields=["DepartmentOption"],
    bind=alt.binding_select(options=dashboard_department_options, name="Department: "),
    value=[{"DepartmentOption": "All Development-Related Departments"}]
)

def make_summary_card(data, label, value_field, width=190, value_size=20, tooltip_field=None):
    base = (
        alt.Chart(data)
        .transform_filter(year_filter)
        .transform_filter(department_filter)
        .properties(width=width, height=70)
    )

    label_text = base.mark_text(
        align="center",
        fontSize=12,
        fontWeight="bold",
        color="#666666"
    ).encode(
        x=alt.value(width / 2),
        y=alt.value(18),
        text=alt.value(label)
    )

    value_text = base.mark_text(
        align="center",
        fontSize=value_size,
        fontWeight="bold",
        color="#1A1A1A"
    ).encode(
        x=alt.value(width / 2),
        y=alt.value(46),
        text=alt.Text(f"{value_field}:N")
    )

    if tooltip_field is not None:
        value_text = value_text.encode(
            tooltip=[alt.Tooltip(f"{tooltip_field}:N", title=label)]
        )

    return (label_text + value_text)

total_spending_card = make_summary_card(
    summary_df,
    "Total Spending",
    "TotalSpendingDisplay",
    width=180,
    value_size=22
)

payment_count_card = make_summary_card(
    summary_df,
    "Number of Payments",
    "PaymentCountDisplay",
    width=170,
    value_size=22
)

top_category_card = make_summary_card(
    summary_df,
    "Top Spending Category",
    "TopCategoryDisplay",
    width=230,
    value_size=13,
    tooltip_field="TopCategoryFull"
)

top_vendor_card = make_summary_card(
    summary_df,
    "Top Vendor / Payee",
    "TopVendorDisplay",
    width=240,
    value_size=13,
    tooltip_field="TopVendorFull"
)

summary_row = alt.hconcat(
    total_spending_card,
    payment_count_card,
    top_category_card,
    top_vendor_card,
    spacing=12
)

category_chart = (
    alt.Chart(category_top10)
    .transform_filter(year_filter)
    .transform_filter(department_filter)
    .mark_bar(color="#4E79A7")
    .encode(
        x=alt.X(
            "TotalSpending:Q",
            title="Total Spending ($)",
            axis=alt.Axis(format="$,.2s")
        ),
        y=alt.Y(
            "Category:N",
            sort="-x",
            title=None,
            axis=alt.Axis(labelLimit=230)
        ),
        tooltip=[
            alt.Tooltip("Year:O", title="Year"),
            alt.Tooltip("DepartmentOption:N", title="Department"),
            alt.Tooltip("Category:N", title="Expense Category"),
            alt.Tooltip("TotalSpending:Q", title="Total Spending", format="$,.2f"),
            alt.Tooltip("PaymentCount:Q", title="Number of Payments", format=",.0f")
        ]
    )
    .properties(
        width=410,
        height=340,
        title="Top Spending Categories"
    )
)

vendor_chart = (
    alt.Chart(vendor_top10)
    .transform_filter(year_filter)
    .transform_filter(department_filter)
    .mark_bar(color="#E17C05")
    .encode(
        x=alt.X(
            "TotalSpending:Q",
            title="Total Spending ($)",
            axis=alt.Axis(format="$,.2s")
        ),
        y=alt.Y(
            "Vendor:N",
            sort="-x",
            title=None,
            axis=alt.Axis(labelLimit=230)
        ),
        tooltip=[
            alt.Tooltip("Year:O", title="Year"),
            alt.Tooltip("DepartmentOption:N", title="Department"),
            alt.Tooltip("Vendor:N", title="Vendor / Payee"),
            alt.Tooltip("TotalSpending:Q", title="Total Spending", format="$,.2f"),
            alt.Tooltip("PaymentCount:Q", title="Number of Payments", format=",.0f")
        ]
    )
    .properties(
        width=410,
        height=340,
        title="Top Vendors / Payees"
    )
)

breakdown_row = alt.hconcat(category_chart, vendor_chart, spacing=24)

development_spending_dashboard = (
    alt.vconcat(summary_row, breakdown_row, spacing=22)
    .add_params(year_filter, department_filter)
    .properties(title="What Drives Development-Related Spending in Urbana?")
    .configure_title(fontSize=18, anchor="middle")
    .configure_axis(labelFontSize=12, titleFontSize=13, gridColor="#E6E6E6")
    .configure_view(stroke="#D9D9D9")
)

development_spending_dashboard


alt.VConcatChart(...)

After comparing development activity and public spending trends over time, this dashboard shifts the analysis from overall patterns to the composition of spending itself. Rather than focusing only on whether spending rises or falls alongside permit activity, the dashboard helps identify where development-related funding is actually going. By filtering by year and department, it becomes possible to examine which spending categories, projects, and vendors account for the largest shares of Community Development and Public Works expenditures. This deeper breakdown helps explain why spending may not directly follow permit counts each year, since development-related expenditures can also be shaped by infrastructure projects, contractual services, maintenance needs, staffing, administrative costs, or long-term planning priorities. In this way, the dashboard extends the project’s central question by investigating what may drive Urbana’s development-related spending beyond permit activity alone.

## Save Charts

In [13]:
bars.save("overall_spending_by_department.html")
development_line.save("development_related_spending_over_time.html")
permits_line.save("community_development_permits_over_time.html")
comparison_chart.save("development_activity_and_public_spending_trends.html")
development_spending_dashboard.save("what_drives_development_related_spending_dashboard.html")